In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime, timezone

BRONZE_SCHEMA = "supplysage_bronze"
SILVER_SCHEMA = "supplysage_silver"

SOURCE_TABLE = f"{BRONZE_SCHEMA}.bronze_retail_inventory"
TARGET_TABLE = f"{SILVER_SCHEMA}.silver_retail_inventory"

TRANSFORM_RUN_ID = f"silver_retail_inventory_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
CREATED_BY = "Vigya"
NOTEBOOK_NAME = "10_silver_retail_inventory"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

print("TRANSFORM_RUN_ID:", TRANSFORM_RUN_ID)
print("SOURCE_TABLE:", SOURCE_TABLE)
print("TARGET_TABLE:", TARGET_TABLE)

bronze_inventory_df = spark.table(SOURCE_TABLE)

print("Bronze retail inventory row count:", bronze_inventory_df.count())

display(bronze_inventory_df.limit(10))
bronze_inventory_df.printSchema()

In [0]:
def clean_string_col(col_name):
    return (
        F.when(F.col(col_name).isNull(), None)
        .when(F.trim(F.col(col_name)) == "", None)
        .when(F.lower(F.trim(F.col(col_name))).isin("null", "none", "nan"), None)
        .otherwise(F.trim(F.col(col_name)))
    )


silver_inventory_df = (
    bronze_inventory_df
    .select(
        F.to_date(F.col("Date")).alias("inventory_date"),
        clean_string_col("Store ID").alias("store_id"),
        clean_string_col("Product ID").alias("product_id"),
        clean_string_col("Category").alias("category"),
        clean_string_col("Region").alias("region"),

        F.col("Inventory Level").cast("int").alias("inventory_level"),
        F.col("Units Sold").cast("int").alias("units_sold"),
        F.col("Units Ordered").cast("int").alias("units_ordered"),

        F.col("Demand Forecast").cast("double").alias("demand_forecast_raw"),
        F.col("Price").cast("double").alias("price"),
        F.col("Discount").cast("double").alias("discount"),

        clean_string_col("Weather Condition").alias("weather_condition"),
        F.col("Holiday/Promotion").cast("int").alias("holiday_promotion_flag"),
        F.col("Competitor Pricing").cast("double").alias("competitor_pricing"),
        clean_string_col("Seasonality").alias("seasonality")
    )
    .withColumn("inventory_date_id", F.date_format(F.col("inventory_date"), "yyyyMMdd").cast("int"))
    .withColumn(
        "demand_forecast_was_negative",
        F.when(F.col("demand_forecast_raw") < 0, F.lit(True)).otherwise(F.lit(False))
    )
    .withColumn(
        "demand_forecast",
        F.when(F.col("demand_forecast_raw").isNull(), None)
        .when(F.col("demand_forecast_raw") < 0, F.lit(0.0))
        .otherwise(F.col("demand_forecast_raw"))
    )
    .withColumn(
        "inventory_position",
        F.coalesce(F.col("inventory_level"), F.lit(0)) + F.coalesce(F.col("units_ordered"), F.lit(0))
    )
    .withColumn(
        "stockout_gap_units",
        F.greatest(F.col("demand_forecast") - F.col("inventory_position"), F.lit(0.0))
    )
    .withColumn(
        "is_stockout_risk",
        F.when(
            (F.col("demand_forecast").isNotNull()) &
            (F.col("inventory_position") < F.col("demand_forecast")),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "inventory_coverage_ratio",
        F.when(F.col("demand_forecast") > 0, F.col("inventory_position") / F.col("demand_forecast"))
        .otherwise(None)
    )
    .withColumn(
        "sell_through_rate",
        F.when(F.col("inventory_level") > 0, F.col("units_sold") / F.col("inventory_level"))
        .otherwise(None)
    )
    .withColumn(
        "price_vs_competitor",
        F.when(F.col("competitor_pricing").isNotNull(), F.col("price") - F.col("competitor_pricing"))
        .otherwise(None)
    )
    .withColumn("inventory_record_id", F.sha2(F.concat_ws("||", "inventory_date", "store_id", "product_id"), 256))
    .withColumn("silver_transform_run_id", F.lit(TRANSFORM_RUN_ID))
    .withColumn("silver_created_at", F.lit(datetime.now(timezone.utc).isoformat()))
    .withColumn("silver_created_by", F.lit(CREATED_BY))
    .withColumn("silver_notebook_name", F.lit(NOTEBOOK_NAME))
)

silver_inventory_row_count = silver_inventory_df.count()

print("Silver retail inventory row count:", silver_inventory_row_count)

display(silver_inventory_df.limit(20))
silver_inventory_df.printSchema()

In [0]:
inventory_validation_rows = []

source_row_count = bronze_inventory_df.count()
target_row_count = silver_inventory_df.count()

inventory_validation_rows.append({
    "validation_check": "row_count_preserved",
    "expected_value": str(source_row_count),
    "actual_value": str(target_row_count),
    "status": "PASS" if source_row_count == target_row_count else "FAIL",
    "issue_detail": None if source_row_count == target_row_count else "Silver inventory row count does not match Bronze row count."
})

null_key_count = (
    silver_inventory_df
    .filter(
        F.col("inventory_date").isNull()
        | F.col("store_id").isNull()
        | F.col("product_id").isNull()
    )
    .count()
)

inventory_validation_rows.append({
    "validation_check": "business_keys_not_null",
    "expected_value": "0 null key rows",
    "actual_value": str(null_key_count),
    "status": "PASS" if null_key_count == 0 else "FAIL",
    "issue_detail": None if null_key_count == 0 else "Null values found in inventory_date, store_id, or product_id."
})

duplicate_grain_count = (
    silver_inventory_df
    .groupBy("inventory_date", "store_id", "product_id")
    .agg(F.count("*").alias("row_count"))
    .filter(F.col("row_count") > 1)
    .count()
)

inventory_validation_rows.append({
    "validation_check": "grain_unique_date_store_product",
    "expected_value": "0 duplicate grain rows",
    "actual_value": str(duplicate_grain_count),
    "status": "PASS" if duplicate_grain_count == 0 else "FAIL",
    "issue_detail": None if duplicate_grain_count == 0 else "Duplicate rows found for inventory_date + store_id + product_id."
})

negative_clean_demand_count = (
    silver_inventory_df
    .filter(F.col("demand_forecast") < 0)
    .count()
)

inventory_validation_rows.append({
    "validation_check": "clean_demand_forecast_non_negative",
    "expected_value": "0 negative cleaned demand forecast rows",
    "actual_value": str(negative_clean_demand_count),
    "status": "PASS" if negative_clean_demand_count == 0 else "FAIL",
    "issue_detail": None if negative_clean_demand_count == 0 else "Cleaned demand_forecast still has negative values."
})

negative_raw_demand_count = (
    silver_inventory_df
    .filter(F.col("demand_forecast_raw") < 0)
    .count()
)

flagged_negative_demand_count = (
    silver_inventory_df
    .filter(F.col("demand_forecast_was_negative") == True)
    .count()
)

inventory_validation_rows.append({
    "validation_check": "negative_demand_forecast_flagged",
    "expected_value": str(negative_raw_demand_count),
    "actual_value": str(flagged_negative_demand_count),
    "status": "PASS" if negative_raw_demand_count == flagged_negative_demand_count else "FAIL",
    "issue_detail": None if negative_raw_demand_count == flagged_negative_demand_count else "Negative raw demand forecast rows were not correctly flagged."
})

invalid_numeric_count = (
    silver_inventory_df
    .filter(
        (F.col("inventory_level") < 0)
        | (F.col("units_sold") < 0)
        | (F.col("units_ordered") < 0)
        | (F.col("price") < 0)
        | (F.col("discount") < 0)
        | (F.col("competitor_pricing") < 0)
    )
    .count()
)

inventory_validation_rows.append({
    "validation_check": "core_numeric_fields_non_negative",
    "expected_value": "0 invalid numeric rows",
    "actual_value": str(invalid_numeric_count),
    "status": "PASS" if invalid_numeric_count == 0 else "FAIL",
    "issue_detail": None if invalid_numeric_count == 0 else "Negative values found in inventory, sales, order, price, discount, or competitor pricing fields."
})

invalid_holiday_flag_count = (
    silver_inventory_df
    .filter(~F.col("holiday_promotion_flag").isin(0, 1))
    .count()
)

inventory_validation_rows.append({
    "validation_check": "holiday_promotion_flag_valid",
    "expected_value": "Only 0 or 1",
    "actual_value": str(invalid_holiday_flag_count),
    "status": "PASS" if invalid_holiday_flag_count == 0 else "FAIL",
    "issue_detail": None if invalid_holiday_flag_count == 0 else "holiday_promotion_flag contains values outside 0 or 1."
})

null_record_id_count = (
    silver_inventory_df
    .filter(F.col("inventory_record_id").isNull())
    .count()
)

inventory_validation_rows.append({
    "validation_check": "inventory_record_id_not_null",
    "expected_value": "0 null inventory_record_id rows",
    "actual_value": str(null_record_id_count),
    "status": "PASS" if null_record_id_count == 0 else "FAIL",
    "issue_detail": None if null_record_id_count == 0 else "Null inventory_record_id values found."
})

inventory_validation_schema = T.StructType([
    T.StructField("validation_check", T.StringType(), True),
    T.StructField("expected_value", T.StringType(), True),
    T.StructField("actual_value", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("issue_detail", T.StringType(), True)
])

inventory_validation_df = spark.createDataFrame(
    inventory_validation_rows,
    schema=inventory_validation_schema
)

display(inventory_validation_df.orderBy("status", "validation_check"))

fail_count = inventory_validation_df.filter(F.col("status") == "FAIL").count()

print("Silver retail inventory validation failures:", fail_count)

In [0]:
if fail_count == 0:
    (
        silver_inventory_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TARGET_TABLE)
    )

    print(f"✅ Wrote Silver retail inventory table: {TARGET_TABLE}")
else:
    raise ValueError("Silver retail inventory validation failed. Fix issues before writing.")

In [0]:
written_inventory_df = spark.table(TARGET_TABLE)

written_row_count = written_inventory_df.count()

print("Written Silver retail inventory row count:", written_row_count)

display(
    written_inventory_df
    .select(
        "inventory_record_id",
        "inventory_date",
        "store_id",
        "product_id",
        "category",
        "region",
        "inventory_level",
        "units_sold",
        "units_ordered",
        "demand_forecast_raw",
        "demand_forecast",
        "demand_forecast_was_negative",
        "inventory_position",
        "stockout_gap_units",
        "is_stockout_risk",
        "inventory_coverage_ratio"
    )
    .limit(20)
)

if written_row_count == 73100:
    print("✅ Read-back check passed: Silver retail inventory table exists and row count is correct.")
else:
    print("❌ Read-back check failed: Row count does not match expected 73,100.")

In [0]:
inventory_validation_results_df = (
    inventory_validation_df
    .withColumn("transform_run_id", F.lit(TRANSFORM_RUN_ID))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("target_table", F.lit(TARGET_TABLE))
    .withColumn("validated_at", F.lit(datetime.now(timezone.utc).isoformat()))
    .withColumn("created_by", F.lit(CREATED_BY))
    .withColumn("notebook_name", F.lit(NOTEBOOK_NAME))
    .select(
        "transform_run_id",
        "source_table",
        "target_table",
        "validation_check",
        "expected_value",
        "actual_value",
        "status",
        "issue_detail",
        "validated_at",
        "created_by",
        "notebook_name"
    )
)

validation_results_table = f"{SILVER_SCHEMA}.silver_transform_validation_results"

try:
    spark.sql(f"""
    DELETE FROM {validation_results_table}
    WHERE transform_run_id = '{TRANSFORM_RUN_ID}'
    """)
except Exception as e:
    print("Validation results table does not exist yet. It will be created now.")

(
    inventory_validation_results_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(validation_results_table)
)

display(
    spark.table(validation_results_table)
    .filter(F.col("transform_run_id") == TRANSFORM_RUN_ID)
    .groupBy("status")
    .agg(F.count("*").alias("validation_check_count"))
    .orderBy("status")
)

print("✅ Notebook 10 PASSED: silver_retail_inventory created and validated.")
print("Next notebook: 11_silver_dataco_orders_shipments")